In [ ]:
# manejo de datos
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

# api Xenocanto
from xenopy import Query
from scipy import signal
import scipy.signal.windows

# manejo de audio
import librosa
import librosa.display

In [ ]:
# Ruta al archivo MP3
ruta_al_archivo = 'prueba.mp3'

# Cargar el archivo MP3
y, sr = librosa.load(ruta_al_archivo)

# 'y' es un array de numpy que contiene los datos de audio
# 'sr' es la frecuencia de muestreo del audio (número de muestras por segundo)

In [ ]:
# Set the hop length; at 22050 Hz, 512 samples ~= 23ms
hop_length = 512

# Separate harmonics and percussives into two waveforms
y_harmonic, y_percussive = librosa.effects.hpss(y)

In [ ]:
# Beat track on the percussive signal
tempo, beat_frames = librosa.beat.beat_track(y=y_percussive, sr=sr)

In [ ]:
# Compute MFCC features from the raw signal
mfcc = librosa.feature.mfcc(y=y, sr=sr, hop_length=hop_length, n_mfcc=13)

In [ ]:
# And the first-order differences (delta features)
mfcc_delta = librosa.feature.delta(mfcc)

In [ ]:
# Stack and synchronize between beat events
# This time, we'll use the mean value (default) instead of median
beat_mfcc_delta = librosa.util.sync(np.vstack([mfcc, mfcc_delta]),
                                    beat_frames)

In [ ]:
# Compute chroma features from the harmonic signal
chromagram = librosa.feature.chroma_cqt(y=y_harmonic, sr=sr)

In [ ]:
# Extraer características de audio
features = {
    'Longitud de onda': len(y),
    'Intensidad media': np.mean(librosa.feature.rms(y=y)),
    'Intensidad máxima': np.max(librosa.feature.rms(y=y)),
    'Tonos principales': len(librosa.core.piptrack(y=y)[1]),
    'Melodía': np.mean(librosa.feature.tonnetz(y=y, sr=sr)),
    'Centroide espectral': np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),
    'Rolloff espectral': np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)),
    'Contraste espectral': np.mean(librosa.feature.spectral_contrast(y=y, sr=sr)),
    'Ancho de banda espectral': np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)),
    'Croma': np.mean(librosa.feature.chroma_stft(y=y, sr=sr)),
    'Tempo': librosa.beat.tempo(y=y, sr=sr)[0],
    'Pulso': np.mean(librosa.feature.tempogram(y=y, sr=sr)),
    'Coeficientes MFCC': np.mean(librosa.feature.mfcc(y=y, sr=sr)),
    'RMS': np.mean(librosa.feature.rms(y=y)),
    'Cens': np.mean(librosa.feature.chroma_cens(y=y, sr=sr)),
    'Piptrack': np.mean(librosa.core.piptrack(y=y)[1]),
    'Cruces por cero': np.mean(librosa.feature.zero_crossing_rate(y)),
    'Cromagrama CQT': np.mean(librosa.feature.chroma_cqt(y=y, sr=sr)),
    'Cromagrama CENS': np.mean(librosa.feature.chroma_cens(y=y, sr=sr)),
    'Melspectrogram': np.mean(librosa.feature.melspectrogram(y=y, sr=sr)),
    'Polynomial coefficients': np.mean(librosa.feature.poly_features(y=y, sr=sr)),
    'Fourier tempogram': np.mean(librosa.feature.fourier_tempogram(y=y, sr=sr)),
    'Ceros Cruzados': np.mean(librosa.feature.zero_crossing_rate(y)),
    'Perceptrual Sharpness': np.mean(librosa.feature.spectral_flatness(y=y)),
    'Loudness': np.sum(librosa.feature.rms(y=y)),
    #'Pulsos': len(librosa.beat.plp(y=y, sr=sr)[1])
} 

# Crear un DataFrame para almacenar los datos
df = pd.DataFrame(features, index=[0])

# Guardar los datos en un archivo CSV
#df.to_csv('datos_de_audio.csv', index=False)
df.head()

In [ ]:
# Aggregate chroma features between beat events
# We'll use the median value of each feature between beat frames
beat_chroma = librosa.util.sync(chromagram,
                                beat_frames,
                                aggregate=np.median)

In [ ]:
# Finally, stack all beat-synchronous features together
beat_features = np.vstack([beat_chroma, beat_mfcc_delta])

In [ ]:
y_harmonic, y_percussive = librosa.effects.hpss(y)

In [ ]:
mfcc = librosa.feature.mfcc(y=y, sr=sr, hop_length=hop_length, n_mfcc=13)

In [ ]:
mfcc_delta = librosa.feature.delta(mfcc)

In [ ]:
beat_mfcc_delta = librosa.util.sync(np.vstack([mfcc, mfcc_delta]),
                                    beat_frames)

In [ ]:
beat_mfcc_delta = librosa.util.sync(np.vstack([mfcc, mfcc_delta]),
                                    beat_frames)

In [ ]:
beat_chroma = librosa.util.sync(chromagram,
                                beat_frames,
                                aggregate=np.median)

In [ ]:
beat_features = np.vstack([beat_chroma, beat_mfcc_delta])